# UNIT IV — Multimodal Generative AI Applications
### Gen AI Lab — 10 Experiments

**Notes before you run this notebook:**
- This notebook is designed to run on **Google Colab** (preferably with a GPU runtime: `Runtime > Change runtime type > T4 GPU`).
- Each experiment is self-contained: install cell(s) + code cell(s) + markdown explanation.
- Pre-trained models are pulled from **Hugging Face Hub** (transformers/diffusers) so no training is required — only inference.
- Run cells **top to bottom**. Installation cells only need to run once per session.
- If a model download is slow/fails, re-run the cell (Hugging Face mirrors can be flaky) or switch to the smaller model suggested in comments.

---
## Experiment 1: College Query Chatbot using a Pre-trained LLM
**Aim:** Develop an AI chatbot that answers student queries related to an engineering college using a pre-trained language model.

**Approach:** We use a pre-trained instruction-tuned model (`google/flan-t5-base`) combined with a small college knowledge base (context) so the model answers grounded in the college's actual info (a simple Retrieval-Augmented Generation style prompt).

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
from transformers import pipeline

chatbot = pipeline("text2text-generation", model="google/flan-t5-base")

# Small college knowledge base (replace with your own college's real data)
college_kb = """
ABC College of Engineering was established in 1998.
It offers B.Tech programs in Computer Science, Electronics, Mechanical, Civil and AI & Data Science.
The college is affiliated to Anna University and approved by AICTE.
Admissions are through TNEA counselling for Tamil Nadu students.
The college has a placement cell with an average package of 4.5 LPA and highest package of 22 LPA.
Hostel facilities are available separately for boys and girls.
The library has over 50,000 books and digital access to IEEE and Springer journals.
The academic year is divided into two semesters: odd (July-Nov) and even (Dec-April).
"""

def ask_college_bot(question, context=college_kb):
    prompt = f"Answer the question using the context below.\nContext: {context}\nQuestion: {question}\nAnswer:"
    result = chatbot(prompt, max_new_tokens=100)
    return result[0]['generated_text']

# Test queries
queries = [
    "When was the college established?",
    "What is the highest placement package offered?",
    "Which university is the college affiliated to?",
    "Does the college have hostel facilities?"
]

for q in queries:
    print(f"Q: {q}\nA: {ask_college_bot(q)}\n")

---
## Experiment 2: Engineering-Support Chatbot using NLP Techniques
**Aim:** Design an engineering-support chatbot that can answer technical questions and provide relevant solutions using NLP techniques.

**Approach:** We use a pre-trained **extractive Question Answering** model (`deepset/roberta-base-squad2`) that reads a technical passage (e.g., troubleshooting manual) and extracts precise answers — a common NLP technique for support bots.

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

technical_doc = """
In reinforced concrete design, the factor of safety is applied to account for uncertainties in loading and material strength.
A common cause of motor overheating in induction motors is voltage imbalance across the three phases.
In control systems, a PID controller uses Proportional, Integral, and Derivative terms to minimize the error signal.
Bandwidth in a network refers to the maximum rate of data transfer across a given path.
In thermodynamics, the first law states that energy cannot be created or destroyed, only converted from one form to another.
A common fix for a CNC machine giving axis position errors is to re-home the machine and check the encoder cables.
"""

def engineering_support_bot(question, context=technical_doc):
    result = qa_pipeline(question=question, context=context)
    return result['answer'], round(result['score'], 3)

questions = [
    "What causes motor overheating in induction motors?",
    "What are the three terms used in a PID controller?",
    "What is the fix for CNC axis position errors?",
    "What does the first law of thermodynamics state?"
]

for q in questions:
    ans, score = engineering_support_bot(q)
    print(f"Q: {q}\nA: {ans}  (confidence: {score})\n")

---
## Experiment 3: Text-to-Image Generation (Engineering Concept)
**Aim:** Generate an engineering-related image, such as a bridge or robotic system, from a suitable text prompt using a pre-trained text-to-image model.

**Approach:** We use **Stable Diffusion** (`runwayml/stable-diffusion-v1-5`) via the `diffusers` library. Requires a GPU runtime for reasonable speed.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe = pipe.to(device)

In [ ]:
prompt = "a futuristic suspension bridge over a river, engineering blueprint style, highly detailed, daylight"

image = pipe(prompt, num_inference_steps=30).images[0]
image.save("bridge.png")
image

---
## Experiment 4: Comparing Images Generated from Different Prompts
**Aim:** Create multiple images from different text prompts and compare how changes in the prompts affect the generated images.

**Approach:** Reuse the Stable Diffusion pipeline from Experiment 3 (run that cell first) and generate images for several related prompts, then display them side-by-side for comparison.

In [ ]:
import matplotlib.pyplot as plt

prompts = [
    "a robotic arm assembling a car on a factory floor, realistic, industrial lighting",
    "a humanoid robot walking in an engineering lab, futuristic, cinematic lighting",
    "a robotic arm assembling a car, cartoon style, colorful",
    "a rusty old robotic arm in an abandoned factory, dark, moody lighting"
]

generated_images = []
for p in prompts:
    img = pipe(p, num_inference_steps=30).images[0]
    generated_images.append(img)

fig, axes = plt.subplots(1, len(prompts), figsize=(20, 5))
for ax, img, p in zip(axes, generated_images, prompts):
    ax.imshow(img)
    ax.set_title(p, fontsize=8, wrap=True)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("\nObservation: Changing style keywords (realistic vs cartoon), lighting (cinematic vs moody),")
print("and condition (new vs rusty/abandoned) significantly alters color palette, mood and rendering style")
print("of the same base subject (robotic arm), demonstrating the sensitivity of diffusion models to prompt wording.")

---
## Experiment 5: Speech-to-Text for Engineering Queries
**Aim:** Develop a Speech-to-Text application that converts a user's spoken engineering-related query into written text using a pre-trained AI model.

**Approach:** We use OpenAI's **Whisper** model (via Hugging Face `openai/whisper-base`) for automatic speech recognition (ASR). You can either upload an audio file or record one in Colab.

In [ ]:
!pip install -q transformers torchaudio soundfile

In [ ]:
# Upload an audio file (wav/mp3) containing a spoken engineering query
from google.colab import files
uploaded = files.upload()  # select your .wav/.mp3 file
audio_path = list(uploaded.keys())[0]

In [ ]:
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-base")

result = asr(audio_path)
print("Transcribed Text:")
print(result["text"])

---
## Experiment 6: Text-to-Speech for Engineering Content
**Aim:** Develop a Text-to-Speech application that converts engineering-related text into natural-sounding speech using a pre-trained AI model.

**Approach:** We use Google's **gTTS** (Google Text-to-Speech) engine, which produces natural-sounding speech from text with minimal setup. (For a fully local pre-trained neural TTS model, `suno/bark` or Microsoft `speecht5_tts` can be swapped in — commented alternative included.)

In [ ]:
!pip install -q gTTS

In [ ]:
from gtts import gTTS
from IPython.display import Audio

engineering_text = """
A truss is a structure made up of straight members connected at joints, primarily used to support loads
over long spans, such as in bridges and roof structures. Each member of a truss experiences either
tension or compression forces, and the design ensures the structure remains stable under expected loads.
"""

tts = gTTS(text=engineering_text, lang="en")
tts.save("engineering_speech.mp3")

Audio("engineering_speech.mp3")

In [ ]:
# --- Optional: Fully pre-trained neural TTS alternative using Hugging Face SpeechT5 ---
# !pip install -q transformers sentencepiece datasets soundfile
# from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
# from datasets import load_dataset
# import torch, soundfile as sf
#
# processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
# model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
# vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
# embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
# speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
#
# inputs = processor(text=engineering_text, return_tensors="pt")
# speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)
# sf.write("speecht5_output.wav", speech.numpy(), samplerate=16000)

---
## Experiment 7: Summarizing Lengthy Engineering Documents
**Aim:** Develop an AI application that summarizes a lengthy engineering document into a short and meaningful summary using a pre-trained language model.

**Approach:** We use the pre-trained **BART** summarization model (`facebook/bart-large-cnn`).

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

long_engineering_doc = """
Reinforced concrete is a composite material in which concrete's relatively low tensile strength and ductility
are compensated for by the inclusion of reinforcement having higher tensile strength or ductility. The
reinforcement is usually steel, in the form of rebar, and is usually embedded passively in the concrete
before it sets. Reinforcing schemes are generally designed to resist tensile stresses in particular regions
of the concrete that might cause unacceptable cracking or structural failure. Modern reinforced concrete
can contain varied reinforcing materials made of steel, polymers or alternate composite material in
conjunction with rebar or not. Reinforced concrete may also be permanently stressed (concrete in
compression, reinforcement in tension), so as to improve the behaviour of the final structure under working
loads. In the United States, the most common methods of doing this are known as pre-tensioning and
post-tensioning. For a strong, ductile and durable construction, the reinforcement needs to have the
following properties at least: high relative strength, high toleration of tensile strain, good bond to the
concrete, thermal compatibility, and durability in the concrete environment, irrespective of corrosion or
sustained stress. Rebar has become standard reinforcing material used in concrete construction worldwide,
and it is usually manufactured from unfinished tempered steel and it has ribbing to promote a better bond
with the concrete.
"""

summary = summarizer(long_engineering_doc, max_length=80, min_length=25, do_sample=False)
print("Original length (words):", len(long_engineering_doc.split()))
print("\nSummary:")
print(summary[0]['summary_text'])
print("\nSummary length (words):", len(summary[0]['summary_text'].split()))

---
## Experiment 8: Machine Translation (English → Indian Language)
**Aim:** Develop a machine translation application that translates an engineering document from English into another Indian language using a pre-trained translation model.

**Approach:** We use the pre-trained **Helsinki-NLP OPUS-MT** English-to-Hindi model. (Other Indian languages like Tamil, Telugu, Malayalam can be used by swapping the model name, e.g. `Helsinki-NLP/opus-mt-en-ta` may not exist for all pairs — `facebook/nllb-200-distilled-600M` is a robust multilingual alternative covering 200+ languages including all major Indian languages, shown as the second option.)

In [ ]:
!pip install -q transformers sentencepiece sacremoses

In [ ]:
from transformers import pipeline

# Option 1: English -> Hindi (Helsinki-NLP)
translator_hi = pipeline("translation", model="Helsinki-NLP/opus-mt-en-hi")

engineering_text_en = (
    "A gear is a rotating machine part having cut teeth which mesh with another toothed part "
    "to transmit torque. Gears are used to change speed, torque, and direction of a power source."
)

translated_hi = translator_hi(engineering_text_en)
print("English:", engineering_text_en)
print("\nHindi Translation:", translated_hi[0]['translation_text'])

In [ ]:
# Option 2: Multilingual NLLB model — works for ANY Indian language (Tamil shown here)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

translator_nllb = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    src_lang="eng_Latn",
    tgt_lang="tam_Taml"  # Tamil. Try hin_Deva (Hindi), tel_Telu (Telugu), mal_Mlym (Malayalam), etc.
)

translated_ta = translator_nllb(engineering_text_en, max_length=200)
print("Tamil Translation:", translated_ta[0]['translation_text'])

---
## Experiment 9: AI-based Resume Screening for an Engineering Job
**Aim:** Develop an AI-based Resume Screening application that analyses candidate resumes and ranks them according to a given engineering job description.

**Approach:** We use a pre-trained **Sentence-Transformer** embedding model (`all-MiniLM-L6-v2`) to embed the job description and each resume, then rank candidates by cosine similarity.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

job_description = """
We are hiring a Mechanical Design Engineer with experience in SolidWorks, AutoCAD, GD&T,
finite element analysis (FEA), and product design for automotive components. Knowledge of
sheet metal design and DFM/DFA principles is a plus.
"""

resumes = {
    "Candidate_A": "Mechanical engineer with 3 years experience in SolidWorks, GD&T and FEA for automotive parts. Worked on DFM reviews.",
    "Candidate_B": "Software developer skilled in Python, Java, and web development. No mechanical design background.",
    "Candidate_C": "Design engineer proficient in AutoCAD and sheet metal design, familiar with product design lifecycle for automotive industry.",
    "Candidate_D": "Civil engineer with expertise in structural analysis, AutoCAD Civil 3D, and site supervision."
}

jd_embedding = model.encode(job_description, convert_to_tensor=True)

scores = {}
for name, resume_text in resumes.items():
    resume_embedding = model.encode(resume_text, convert_to_tensor=True)
    similarity = util.cos_sim(jd_embedding, resume_embedding).item()
    scores[name] = round(similarity, 4)

ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("Resume Ranking (higher score = better match to job description):\n")
for rank, (name, score) in enumerate(ranked, start=1):
    print(f"{rank}. {name} — similarity score: {score}")

---
## Experiment 10: AI-based Research Assistant
**Aim:** Develop an AI-based Research Assistance application that accepts a research topic and generates relevant information, keywords, and a concise summary.

**Approach:** We fetch background information from **Wikipedia**, extract **keywords** using KeyBERT (built on pre-trained sentence embeddings), and generate a **concise summary** using the pre-trained BART summarizer from Experiment 7.

In [ ]:
!pip install -q wikipedia keybert transformers

In [ ]:
import wikipedia
from keybert import KeyBERT
from transformers import pipeline

kw_model = KeyBERT()
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def research_assistant(topic, sentences=8, num_keywords=8):
    # 1. Retrieve information
    try:
        content = wikipedia.summary(topic, sentences=sentences)
    except wikipedia.exceptions.DisambiguationError as e:
        content = wikipedia.summary(e.options[0], sentences=sentences)

    # 2. Extract keywords
    keywords = kw_model.extract_keywords(content, keyphrase_ngram_range=(1, 2),
                                          stop_words="english", top_n=num_keywords)

    # 3. Generate concise summary
    word_count = len(content.split())
    max_len = min(80, max(20, word_count // 2))
    summary = summarizer(content, max_length=max_len, min_length=15, do_sample=False)

    return {
        "topic": topic,
        "raw_information": content,
        "keywords": [kw for kw, score in keywords],
        "concise_summary": summary[0]['summary_text']
    }

result = research_assistant("Finite element method")

print("TOPIC:", result["topic"])
print("\nRAW INFORMATION:\n", result["raw_information"])
print("\nKEYWORDS:\n", result["keywords"])
print("\nCONCISE SUMMARY:\n", result["concise_summary"])

---
## End of Unit IV Lab Experiments

**Models used (all pre-trained, inference-only):**
1. `google/flan-t5-base` — instruction-tuned text generation
2. `deepset/roberta-base-squad2` — extractive QA
3–4. `runwayml/stable-diffusion-v1-5` — text-to-image diffusion
5. `openai/whisper-base` — automatic speech recognition
6. `gTTS` (+ optional `microsoft/speecht5_tts`) — text-to-speech
7. `facebook/bart-large-cnn` — abstractive summarization
8. `Helsinki-NLP/opus-mt-en-hi`, `facebook/nllb-200-distilled-600M` — machine translation
9. `sentence-transformers/all-MiniLM-L6-v2` — semantic similarity ranking
10. `wikipedia` API + `KeyBERT` + `facebook/bart-large-cnn` — research assistant pipeline